# Carga de datos

En este notebook realizamos la carga inicial de la base de datos del Proyecto Integrador del Módulo 5. El objetivo es comprobar que el archivo puede leerse correctamente, revisar su estructura general e identificar condiciones iniciales relevantes antes de continuar con el análisis exploratorio y la ingeniería de características.

Esta etapa no modifica los datos. Su función es validar la lectura del archivo, conocer sus dimensiones, revisar columnas disponibles, detectar valores nulos y duplicados, y clasificar las variables de forma preliminar.

In [27]:
import pandas as pd
from pathlib import Path
from IPython.display import display

## 1. Lectura del archivo

Primero definimos la ruta del archivo `Base_de_datos.csv`. El notebook está ubicado dentro de `src/`, por lo que la ruta principal esperada es `../Base_de_datos.csv`. También se incluye una ruta alternativa para poder ejecutar el notebook desde la raíz del proyecto sin modificar el código.

In [28]:
rutas_posibles = [
    Path("../Base_de_datos.csv"),
    Path("Base_de_datos.csv"),
]

ruta_csv = None

for ruta in rutas_posibles:
    if ruta.exists():
        ruta_csv = ruta
        break

if ruta_csv is None:
    raise FileNotFoundError("No se encontró Base_de_datos.csv en la raíz del proyecto ni en la ruta superior.")

print(f"Archivo cargado desde: {ruta_csv}")

df = pd.read_csv(ruta_csv)
display(df.head())

Archivo cargado desde: ..\Base_de_datos.csv


,tipo_credito,fecha_prestamo,capital_prestado,plazo_meses,edad_cliente,tipo_laboral,salario_cliente,total_otros_prestamos,cuota_pactada,puntaje,...,saldo_mora,saldo_total,saldo_principal,saldo_mora_codeudor,creditos_sectorFinanciero,creditos_sectorCooperativo,creditos_sectorReal,promedio_ingresos_datacredito,tendencia_ingresos,Pago_atiempo
0,7,2024-12-21 11:31:35,3692160.0,10,42,Independiente,8000000,2500000,341296,88.768094,...,0.0,51258.0,51258.0,0.0,5,0,0,908526.0,Estable,1
1,4,2025-04-22 09:47:35,840000.0,6,60,Empleado,3000000,2000000,124876,95.227787,...,0.0,8673.0,8673.0,0.0,0,0,2,939017.0,Creciente,1
2,9,2026-01-08 12:22:40,5974028.4,10,36,Independiente,4036000,829000,529554,47.613894,...,0.0,18702.0,18702.0,0.0,3,0,0,NaN,NaN,0
3,4,2025-08-04 12:04:10,1671240.0,6,48,Empleado,1524547,498000,252420,95.227787,...,0.0,15782.0,15782.0,0.0,3,0,0,1536193.0,Creciente,1
4,9,2025-04-26 11:24:26,2781636.0,11,44,Empleado,5000000,4000000,217037,95.227787,...,0.0,204804.0,204804.0,0.0,3,0,1,933473.0,Creciente,1


Después de cargar la base, revisamos cuántos registros y variables contiene el dataset. Esto permite tener una primera idea del tamaño de la información disponible para el análisis y el modelado posterior.

In [29]:
filas, columnas = df.shape
print(f"Filas: {filas}")
print(f"Columnas: {columnas}")

Filas: 10763
Columnas: 23


También revisamos el nombre de las columnas para identificar las variables disponibles y confirmar la presencia de la variable objetivo del proyecto.

In [30]:
columnas_df = pd.DataFrame({
    "columna": df.columns,
    "tipo_dato": [str(tipo) for tipo in df.dtypes.values]
})

display(columnas_df)

,columna,tipo_dato
0,tipo_credito,int64
1,fecha_prestamo,str
2,capital_prestado,float64
3,plazo_meses,int64
4,edad_cliente,int64
5,tipo_laboral,str
6,salario_cliente,int64
7,total_otros_prestamos,int64
8,cuota_pactada,int64
9,puntaje,float64


## 2. Información general del dataset

Ahora revisamos la estructura general de la base de datos: tipos de variables, cantidad de valores no nulos y composición inicial del DataFrame. Esta revisión permite detectar posibles problemas antes de avanzar al análisis exploratorio detallado.

In [31]:
resumen_df = pd.DataFrame({
    "columna": df.columns,
    "tipo_dato": df.dtypes.astype(str).values,
    "no_nulos": df.notnull().sum().values,
    "nulos": df.isnull().sum().values,
    "porcentaje_nulos": (df.isnull().sum().values / len(df) * 100).round(2)
})

display(resumen_df)

,columna,tipo_dato,no_nulos,nulos,porcentaje_nulos
0,tipo_credito,int64,10763,0,0.00
1,fecha_prestamo,str,10763,0,0.00
2,capital_prestado,float64,10763,0,0.00
3,plazo_meses,int64,10763,0,0.00
4,edad_cliente,int64,10763,0,0.00
5,tipo_laboral,str,10763,0,0.00
6,salario_cliente,int64,10763,0,0.00
7,total_otros_prestamos,int64,10763,0,0.00
8,cuota_pactada,int64,10763,0,0.00
9,puntaje,float64,10763,0,0.00


#### Registros nulos

Los valores nulos pueden afectar el análisis exploratorio y el entrenamiento de modelos, por lo que es necesario identificarlos desde la etapa inicial de carga de datos.

In [32]:
nulos_df = df.isnull().sum().reset_index()
nulos_df.columns = ["columna", "nulos"]
nulos_df["porcentaje"] = (nulos_df["nulos"] / len(df) * 100).round(2)

nulos_df = nulos_df[nulos_df["nulos"] > 0].sort_values(
    by="nulos",
    ascending=False
)

if nulos_df.empty:
    print("No se identificaron valores nulos en el dataset.")
else:
    display(nulos_df)

,columna,nulos,porcentaje
21,tendencia_ingresos,2932,27.24
20,promedio_ingresos_datacredito,2930,27.22
16,saldo_mora_codeudor,590,5.48
15,saldo_principal,405,3.76
13,saldo_mora,156,1.45
14,saldo_total,156,1.45
10,puntaje_datacredito,6,0.06


De acuerdo con la revisión de valores nulos realizada anteriormente, algunas variables presentan datos faltantes en proporciones relevantes. Estas columnas deberán ser consideradas en la etapa de ingeniería de características para definir un tratamiento adecuado.

Por ahora no se modifican los valores faltantes, ya que el objetivo de este notebook es validar la carga de datos y comprender su estado inicial antes de aplicar transformaciones.

#### Registros duplicados

También revisamos si existen registros duplicados completos dentro de la base, ya que podrían alterar los resultados del análisis o sesgar el entrenamiento del modelo.

In [33]:
duplicados = df.duplicated().sum()
print(f"Registros duplicados completos: {duplicados}")

Registros duplicados completos: 0


No se identificaron registros duplicados completos en la base de datos. Esto indica que, al menos a nivel de filas completas, no hay repeticiones que deban eliminarse antes del análisis exploratorio.

## 3. Clasificación inicial de variables

Como parte de la carga inicial, separamos las columnas según su tipo de dato. Esto permite distinguir entre variables numéricas, categóricas y de fecha, lo cual será útil para el análisis exploratorio y la ingeniería de características.

In [34]:
columnas_fecha = [col for col in df.columns if "fecha" in col.lower()]
columnas_numericas = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
columnas_categoricas = df.select_dtypes(include=["object"]).columns.tolist()

clasificacion_variables = pd.DataFrame({
    "tipo_variable": ["numérica", "categórica", "fecha"],
    "cantidad": [
        len(columnas_numericas),
        len(columnas_categoricas),
        len(columnas_fecha)
    ],
    "columnas": [
        ", ".join(columnas_numericas),
        ", ".join(columnas_categoricas),
        ", ".join(columnas_fecha)
    ]
})

display(clasificacion_variables)

C:\Users\Ismael2\AppData\Local\Temp\ipykernel_21900\1048390299.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  columnas_categoricas = df.select_dtypes(include=["object"]).columns.tolist()


,tipo_variable,cantidad,columnas
0,numérica,20,"tipo_credito, capital_prestado, plazo_meses, e..."
1,categórica,3,"fecha_prestamo, tipo_laboral, tendencia_ingresos"
2,fecha,1,fecha_prestamo


La variable objetivo `Pago_atiempo` se identifica dentro del conjunto de datos, pero su análisis detallado se desarrolla en el notebook `comprension_eda.ipynb`, donde corresponde realizar la exploración de la variable objetivo y sus relaciones con el resto de variables.

#### Descripción estadística inicial

Finalmente, obtenemos una primera descripción estadística de las variables numéricas. Esta revisión no sustituye al EDA, pero permite observar rangos, promedios y posibles valores extremos de manera preliminar.

In [35]:
display(df.describe().T)

,count,mean,std,min,25%,50%,75%,max
tipo_credito,10763.0,5.411131e+00,2.338279e+00,4.00000,4.000000e+00,4.000000e+00,9.000000e+00,6.800000e+01
capital_prestado,10763.0,2.434315e+06,1.909643e+06,360000.00000,1.224831e+06,1.921920e+06,3.084840e+06,4.144415e+07
plazo_meses,10763.0,1.057558e+01,6.632082e+00,2.00000,6.000000e+00,1.000000e+01,1.200000e+01,9.000000e+01
edad_cliente,10763.0,4.394862e+01,1.506088e+01,19.00000,3.300000e+01,4.200000e+01,5.300000e+01,1.230000e+02
salario_cliente,10763.0,1.721643e+07,3.554767e+08,0.00000,2.000000e+06,3.000000e+06,4.875808e+06,2.200000e+10
total_otros_prestamos,10763.0,6.238870e+06,1.184183e+08,0.00000,5.000000e+05,1.000000e+06,2.000000e+06,6.787675e+09
cuota_pactada,10763.0,2.436174e+05,2.104937e+05,23944.00000,1.210415e+05,1.828630e+05,2.878335e+05,3.816752e+06
puntaje,10763.0,9.117004e+01,1.646544e+01,-38.00999,9.522779e+01,9.522779e+01,9.522779e+01,9.522779e+01
puntaje_datacredito,10757.0,7.807908e+02,1.048780e+02,-7.00000,7.570000e+02,7.910000e+02,8.250000e+02,9.990000e+02
cant_creditosvigentes,10763.0,5.726749e+00,3.977162e+00,0.00000,3.000000e+00,5.000000e+00,8.000000e+00,6.200000e+01


## 4. Conclusiones preliminares

Con esta primera revisión identificamos las dimensiones del dataset, las columnas disponibles, los tipos de datos, la presencia de valores nulos, la ausencia de registros duplicados y la variable objetivo `Pago_atiempo`.

Este notebook se mantiene enfocado en la carga y validación inicial de los datos. El análisis exploratorio detallado, incluyendo la distribución de la variable objetivo y sus relaciones con otras variables, se desarrolla posteriormente en `comprension_eda.ipynb`.

Con esta validación inicial, el proyecto queda preparado para continuar con la etapa de comprensión exploratoria de datos, ingeniería de características y modelamiento supervisado.